# YOLO26n Transfer Learning — Two-Stage Pipeline on NEU-DET

## Goal
Apply **professional two-stage transfer learning** to YOLO26n for steel surface defect detection on NEU-DET (6 classes).

**Target:** Beat 81.0% mAP@0.5 (YOLO26n full fine-tune baseline) and 78.8% (YOLO11n baseline).

## Two-Stage Strategy

| Stage | What | Frozen | Trainable | LR | Epochs |
|-------|------|--------|-----------|-----|--------|
| **1: Feature Extraction** | Load COCO → freeze all → train head | 0 to N-1 | N (Detect head) | 0.001 | 100 |
| **2: Fine-Tuning** | Load S1 → unfreeze top → low LR | 0 to K-1 | K to N (neck+head) | 0.0001 | 300 |

## Why This Works
- **COCO pretrained features** (edges, textures, shapes) are universal — they transfer well to steel defects
- **Stage 1** teaches only the detection head: "which pattern = which defect class"
- **Stage 2** adapts the neck (multi-scale fusion) to defect-specific scales while preserving low-level features
- **Low LR in Stage 2** prevents catastrophic forgetting of useful pretrained weights

## Architecture Overview (YOLO26n)
```
BACKBONE (Early Layers):   Low-level features (edges, textures, gradients) → FREEZE in both stages
NECK     (Middle Layers):  Mid-level features (multi-scale fusion)         → FREEZE in S1, TRAIN in S2
HEAD     (Top Layers):     High-level features (classification, bbox)      → TRAIN in both stages
```

## Evaluation
- All results on **TEST set** (not train or val)
- Train/val loss plots for overfitting detection
- Comparison: Stage 1 vs Stage 2 vs YOLO26n Full vs YOLO11n Baseline

In [6]:
# ============================================================
# Cell 1: Environment Setup & Imports
# ============================================================
import subprocess, sys

# Upgrade ultralytics for YOLO26 support
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'ultralytics>=8.4.0'])
print("Ultralytics upgraded. Restart kernel if prompted.")

import torch
import torch.nn as nn
import ultralytics
from ultralytics import YOLO
from pathlib import Path
import json
import yaml
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Reproducibility ──
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Environment Report ──
print("=" * 60)
print("  ENVIRONMENT")
print("=" * 60)
print(f"  Python:       {sys.version.split()[0]}")
print(f"  PyTorch:      {torch.__version__}")
print(f"  Ultralytics:  {ultralytics.__version__}")
print(f"  CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU:          {torch.cuda.get_device_name(0)}")
    print(f"  VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  WARNING: No GPU — training will be very slow!")

# Verify YOLO26 support
try:
    _test = YOLO("yolo26n.yaml")
    print(f"  YOLO26:       Available")
    del _test
except Exception as e:
    print(f"  YOLO26:       NOT available — {e}")
    print("  Try restarting the kernel after upgrade.")

print("=" * 60)

Ultralytics upgraded. Restart kernel if prompted.
  ENVIRONMENT
  Python:       3.11.15
  PyTorch:      2.6.0+cu124
  Ultralytics:  8.4.83
  CUDA:         True
  GPU:          NVIDIA RTX 2000 Ada Generation
  VRAM:         17.2 GB
  YOLO26:       Available


In [7]:
# ============================================================
# Cell 2: Path Configuration & Data Verification
# ============================================================
from pathlib import Path
import yaml

# ── Paths ──
ROOT = Path(r"D:/DigiSteel-Yolo/DigiSteel-YOLO")
DATA_YAML = ROOT / "configs" / "data" / "neu_det.yaml"
RUNS_DIR = ROOT / "runs" / "detect"
EVALS_DIR = ROOT / "evals"
EVALS_DIR.mkdir(parents=True, exist_ok=True)

# Stage 1 paths
S1_RUN_NAME = "yolov26n_tl_stage1"
S1_BEST_PT = RUNS_DIR / S1_RUN_NAME / "weights" / "best.pt"
S1_RESULTS_CSV = RUNS_DIR / S1_RUN_NAME / "results.csv"

# Stage 2 paths
S2_RUN_NAME = "yolov26n_tl_stage2"
S2_BEST_PT = RUNS_DIR / S2_RUN_NAME / "weights" / "best.pt"
S2_RESULTS_CSV = RUNS_DIR / S2_RUN_NAME / "results.csv"

# ── Verify Dataset ──
print("=" * 60)
print("  DATA VERIFICATION")
print("=" * 60)

errors = []
if not DATA_YAML.exists():
    errors.append(f"Data YAML not found: {DATA_YAML}")
else:
    print(f"  Data YAML: {DATA_YAML}")
    with open(DATA_YAML) as f:
        data_cfg = yaml.safe_load(f)
    raw_path = data_cfg.get("path", "")
    base_path = Path(raw_path) if Path(raw_path).is_absolute() else ROOT / raw_path
    for split in ["train", "val", "test"]:
        split_path = base_path / data_cfg.get(split, "")
        if split_path.exists():
            n = len(list(split_path.glob("*.jpg"))) + len(list(split_path.glob("*.png")))
            print(f"  {split:>5}: {n} images")
        else:
            errors.append(f"{split} not found: {split_path}")

if errors:
    for e in errors:
        print(f"  ERROR: {e}")
    raise FileNotFoundError("Critical paths missing.")

print(f"\n  Stage 1 output: {S1_BEST_PT}")
print(f"  Stage 2 output: {S2_BEST_PT}")
print("=" * 60)

  DATA VERIFICATION
  Data YAML: D:\DigiSteel-Yolo\DigiSteel-YOLO\configs\data\neu_det.yaml
  train: 1290 images
    val: 344 images
   test: 166 images

  Stage 1 output: D:\DigiSteel-Yolo\DigiSteel-YOLO\runs\detect\yolov26n_tl_stage1\weights\best.pt
  Stage 2 output: D:\DigiSteel-Yolo\DigiSteel-YOLO\runs\detect\yolov26n_tl_stage2\weights\best.pt


In [8]:
# ============================================================
# Cell 3: Load YOLO26n Pretrained Model
# ============================================================
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

total_params = sum(p.numel() for p in model.model.parameters())
total_layers = len(list(model.model.model.children()))

print("=" * 60)
print("  YOLO26n PRETRAINED MODEL")
print("=" * 60)
print(f"  Architecture:  YOLO26n (COCO pretrained)")
print(f"  Model class:   {type(model.model).__name__}")
print(f"  Task:          {model.task}")
print(f"  Parameters:    {total_params:,}")
print(f"  Top layers:    {total_layers}")
print(f"  NMS-free:      Yes")
print(f"  DFL-free:      Yes")
print("=" * 60)

  YOLO26n PRETRAINED MODEL
  Architecture:  YOLO26n (COCO pretrained)
  Model class:   DetectionModel
  Task:          detect
  Parameters:    2,572,280
  Top layers:    24
  NMS-free:      Yes
  DFL-free:      Yes


In [9]:
# ============================================================
# Cell 4: Model Summary — Layer-by-Layer Analysis
# ============================================================
from ultralytics import YOLO
import pandas as pd

model = YOLO("yolo26n.pt")

# Print Ultralytics model summary
print("=" * 70)
print("  YOLO26n — MODEL SUMMARY (Ultralytics)")
print("=" * 70)
model.info(verbose=True)

# Build layer table
print(f"\n{'=' * 70}")
print("  LAYER-BY-LAYER TABLE")
print(f"{'=' * 70}\n")

layers = list(model.model.model.children())
total_layers = len(layers)

rows = []
backbone_p = neck_p = head_p = 0
backbone_l = neck_l = head_l = 0

# Detect the Detect layer index
detect_idx = total_layers - 1
for i, layer in enumerate(layers):
    if type(layer).__name__ == "Detect":
        detect_idx = i
        break

# Neck starts after backbone (typically after SPPF/C2PSA)
# For YOLO26: backbone ~0-10, neck ~11 to detect_idx-1, head = detect_idx
neck_start = min(11, detect_idx)

for i, layer in enumerate(layers):
    ltype = type(layer).__name__
    nparams = sum(p.numel() for p in layer.parameters())
    nsub = sum(1 for _ in layer.modules()) - 1

    if i >= detect_idx:
        cat = "HEAD (high-level)"
        head_p += nparams; head_l += nsub
    elif i >= neck_start:
        cat = "NECK (mid-level)"
        neck_p += nparams; neck_l += nsub
    else:
        cat = "BACKBONE (low-level)"
        backbone_p += nparams; backbone_l += nsub

    rows.append({"Idx": i, "Type": ltype, "Params": nparams, "Sub-layers": nsub, "Category": cat})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# Summary table
total_l = backbone_l + neck_l + head_l
total_p = backbone_p + neck_p + head_p

print(f"\n{'=' * 70}")
print("  ARCHITECTURE SUMMARY")
print(f"{'=' * 70}")
if total_l > 0 and total_p > 0:
    print(f"\n  {'Section':<30} {'Layers':>8} {'%':>8} {'Params':>12} {'% Params':>8}")
    print(f"  {'-' * 66}")
    print(f"  {'BACKBONE (low-level)':<30} {backbone_l:>8} {backbone_l/total_l*100:>7.1f}% {backbone_p:>12,} {backbone_p/total_p*100:>7.1f}%")
    print(f"  {'NECK (mid-level)':<30} {neck_l:>8} {neck_l/total_l*100:>7.1f}% {neck_p:>12,} {neck_p/total_p*100:>7.1f}%")
    print(f"  {'HEAD (high-level)':<30} {head_l:>8} {head_l/total_l*100:>7.1f}% {head_p:>12,} {head_p/total_p*100:>7.1f}%")
    print(f"  {'-' * 66}")
    print(f"  {'TOTAL':<30} {total_l:>8} {'100.0%':>8} {total_p:>12,} {'100.0%':>8}")
else:
    print("  WARNING: No layers or parameters found.")

# Store key indices for later cells
print(f"\n  Key indices:")
print(f"    Backbone:  layers 0-{neck_start-1}")
print(f"    Neck:      layers {neck_start}-{detect_idx-1}")
print(f"    Head:      layer  {detect_idx} (Detect)")
print(f"{'=' * 70}")

  YOLO26n — MODEL SUMMARY (Ultralytics)
YOLO26n summary: 260 layers, 2,572,280 parameters, 0 gradients, 6.1 GFLOPs

  LAYER-BY-LAYER TABLE

 Idx     Type  Params  Sub-layers             Category
   0     Conv     464           3 BACKBONE (low-level)
   1     Conv    4672           3 BACKBONE (low-level)
   2     C3k2    6640          15 BACKBONE (low-level)
   3     Conv   36992           3 BACKBONE (low-level)
   4     C3k2   26080          15 BACKBONE (low-level)
   5     Conv  147712           3 BACKBONE (low-level)
   6     C3k2   87040          33 BACKBONE (low-level)
   7     Conv  295424           3 BACKBONE (low-level)
   8     C3k2  346112          33 BACKBONE (low-level)
   9     SPPF  164608           9 BACKBONE (low-level)
  10    C2PSA  249728          30 BACKBONE (low-level)
  11 Upsample       0           0     NECK (mid-level)
  12   Concat       0           0     NECK (mid-level)
  13     C3k2  119808          33     NECK (mid-level)
  14 Upsample       0           0  

---
## Stage 1: Feature Extraction

**Strategy:** Freeze ALL backbone + neck layers, train ONLY the detection head

**Why:** COCO pretrained features (edges, textures) are universal. Only the head needs to learn: "which pattern = which defect class"

**LR:** 0.001 (standard — head needs to learn fast) | **Epochs:** 100

In [10]:
# ============================================================
# Cell 5: Stage 1 — Freeze All Layers & Train Head
# ============================================================
from ultralytics import YOLO
import time, json

print("=" * 70)
print("  STAGE 1: FEATURE EXTRACTION")
print("  Freeze: ALL (backbone + neck) | Train: HEAD (Detect)")
print("=" * 70)

# Load fresh model
s1_model = YOLO("yolo26n.pt")
layers = list(s1_model.model.model.children())
detect_idx = len(layers) - 1
for i, layer in enumerate(layers):
    if type(layer).__name__ == "Detect":
        detect_idx = i
        break

# Freeze all except detection head
frozen_params = 0
trainable_params = 0

for name, param in s1_model.model.named_parameters():
    # Check if this param belongs to the Detect layer
    parts = name.split(".")
    if parts[0] == "model" and len(parts) > 1:
        try:
            layer_idx = int(parts[1])
        except ValueError:
            layer_idx = -1
        if layer_idx >= detect_idx:
            param.requires_grad = True
            trainable_params += param.numel()
        else:
            param.requires_grad = False
            frozen_params += param.numel()
    else:
        param.requires_grad = True
        trainable_params += param.numel()

total_p = frozen_params + trainable_params
print(f"\n  Frozen:     {frozen_params:>12,} params ({frozen_params/total_p*100:.1f}%)")
print(f"  Trainable:  {trainable_params:>12,} params ({trainable_params/total_p*100:.1f}%)")

# Verify per-layer status
print(f"\n  {'Idx':<5} {'Type':<20} {'Params':>12} {'Status':>12}")
print(f"  {'-' * 55}")
for i, layer in enumerate(layers):
    ltype = type(layer).__name__
    nparams = sum(p.numel() for p in layer.parameters())
    is_frozen = all(not p.requires_grad for p in layer.parameters()) if list(layer.parameters()) else True
    status = "FROZEN" if is_frozen else "TRAINABLE"
    print(f"  {i:<5} {ltype:<20} {nparams:>12,} {status:>12}")

# ── Training ──
s1_overrides = {
    "data": str(DATA_YAML),
    "task": "detect",
    "epochs": 100,
    "patience": 30,
    "batch": 16,
    "imgsz": 800,
    "device": 0,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "mosaic": 0.0,
    "mixup": 0.15,
    "degrees": 10.0,
    "translate": 0.2,
    "scale": 0.6,
    "shear": 5.0,
    "perspective": 0.0,
    "flipud": 0.5,
    "fliplr": 0.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "erasing": 0.4,
    "cos_lr": True,
    "deterministic": True,
    "close_mosaic": 10,
    "amp": True,
    "seed": 42,
    "workers": 8,
    "save": True,
    "save_period": 25,
    "plots": True,
    "project": str(RUNS_DIR),
    "name": S1_RUN_NAME,
    "exist_ok": True,
    "verbose": True,
}

print(f"\n  Starting training...")
print(f"  Epochs: {s1_overrides['epochs']} | LR: {s1_overrides['lr0']} | Patience: {s1_overrides['patience']}")

s1_start = time.time()
s1_results = s1_model.train(**s1_overrides)
s1_time = time.time() - s1_start

# Save status
with open(EVALS_DIR / "yolov26n_tl_stage1_status.json", "w") as f:
    json.dump({"training_time_seconds": s1_time, "frozen_params": frozen_params, "trainable_params": trainable_params}, f, indent=2)

print(f"\n{'=' * 60}")
print(f"  STAGE 1 COMPLETE")
print(f"  Duration: {s1_time/3600:.1f} hours")
print(f"  Weights:  {S1_BEST_PT}")
print(f"{'=' * 60}")

  STAGE 1: FEATURE EXTRACTION
  Freeze: ALL (backbone + neck) | Train: HEAD (Detect)

  Frozen:        2,262,624 params (88.0%)
  Trainable:       309,656 params (12.0%)

  Idx   Type                       Params       Status
  -------------------------------------------------------
  0     Conv                          464       FROZEN
  1     Conv                        4,672       FROZEN
  2     C3k2                        6,640       FROZEN
  3     Conv                       36,992       FROZEN
  4     C3k2                       26,080       FROZEN
  5     Conv                      147,712       FROZEN
  6     C3k2                       87,040       FROZEN
  7     Conv                      295,424       FROZEN
  8     C3k2                      346,112       FROZEN
  9     SPPF                      164,608       FROZEN
  10    C2PSA                     249,728       FROZEN
  11    Upsample                        0       FROZEN
  12    Concat                          0       FROZEN
 

In [11]:
# ============================================================
# Cell 6: Stage 1 — Evaluate on Test Set
# ============================================================
from ultralytics import YOLO
import json

print("=" * 70)
print("  STAGE 1: TEST SET EVALUATION")
print("=" * 70)

if not S1_BEST_PT.exists():
    print(f"  ERROR: Weights not found: {S1_BEST_PT}")
    print("  Run Cell 5 first.")
else:
    s1_best = YOLO(str(S1_BEST_PT))

    print(f"\n  Evaluating on TEST set...")
    s1_val = s1_best.val(
        data=str(DATA_YAML), split="test", imgsz=800,
        batch=16, device=0, plots=True, verbose=True,
    )

    s1_map50 = float(s1_val.box.map50)
    s1_map5095 = float(s1_val.box.map)
    s1_prec = float(s1_val.box.mp)
    s1_rec = float(s1_val.box.mr)

    class_names = s1_best.names
    s1_per_class = {}
    for cid, cname in class_names.items():
        if cid < len(s1_val.box.ap50):
            s1_per_class[cname] = float(s1_val.box.ap50[cid])

    print(f"\n{'=' * 60}")
    print("  STAGE 1 TEST RESULTS")
    print(f"{'=' * 60}")
    print(f"  mAP@0.5:      {s1_map50:.4f} ({s1_map50*100:.1f}%)")
    print(f"  mAP@0.5:0.95: {s1_map5095:.4f} ({s1_map5095*100:.1f}%)")
    print(f"  Precision:    {s1_prec:.4f} ({s1_prec*100:.1f}%)")
    print(f"  Recall:       {s1_rec:.4f} ({s1_rec*100:.1f}%)")
    print(f"\n  Per-class AP@0.5:")
    for cname in sorted(s1_per_class):
        bar = "█" * int(s1_per_class[cname] * 40)
        print(f"    {cname:<18} {s1_per_class[cname]*100:.1f}% {bar}")

    # Compare with baselines
    baseline_path = EVALS_DIR / "fresh_baseline_results.json"
    yolov26_full_path = EVALS_DIR / "yolov26n_neudet_results.json"

    if baseline_path.exists():
        with open(baseline_path) as f:
            bl = json.load(f)
        print(f"\n  vs YOLO11n Baseline (78.8%):")
        d = (s1_map50 - bl["map50"]) * 100
        print(f"    mAP@0.5: {bl['map50']*100:.1f}% → {s1_map50*100:.1f}%  ({'+' if d>0 else ''}{d:.1f}%)")

    if yolov26_full_path.exists():
        with open(yolov26_full_path) as f:
            y26 = json.load(f)
        print(f"\n  vs YOLO26n Full Fine-tune (81.0%):")
        d = (s1_map50 - y26["map50"]) * 100
        print(f"    mAP@0.5: {y26['map50']*100:.1f}% → {s1_map50*100:.1f}%  ({'+' if d>0 else ''}{d:.1f}%)")

    # Save
    s1_eval = {
        "stage": "feature_extraction",
        "model": "yolo26n.pt",
        "mAP50": s1_map50,
        "mAP50_95": s1_map5095,
        "precision": s1_prec,
        "recall": s1_rec,
        "per_class_ap50": s1_per_class,
    }
    with open(EVALS_DIR / "yolov26n_tl_stage1_results.json", "w") as f:
        json.dump(s1_eval, f, indent=2)
    print(f"\n  Saved: {EVALS_DIR / 'yolov26n_tl_stage1_results.json'}")

print("\n" + "=" * 70)

  STAGE 1: TEST SET EVALUATION

  Evaluating on TEST set...
Ultralytics 8.4.83  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 0.80.2 MB/s, size: 18.9 KB)
val: Scanning D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\labels\test.cache... 166 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 166/166  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 2.5it/s 4.4s0.2s
                   all        166        371       0.64      0.732      0.744      0.423
               crazing         24         54      0.514      0.373      0.429       0.16
             inclusion         33         77      0.651      0.844      0.816       0.45
               patches         31         80      0.764      0.825       0.86      0.549
        pitted_surface         

In [12]:
# ============================================================
# Cell 7: Stage 1 — Training Curves & Overfitting Analysis
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd

print("=" * 70)
print("  STAGE 1: TRAINING CURVES")
print("=" * 70)

if not S1_RESULTS_CSV.exists():
    print(f"  ERROR: results.csv not found: {S1_RESULTS_CSV}")
else:
    df = pd.read_csv(S1_RESULTS_CSV)
    df.columns = df.columns.str.strip()
    print(f"  Loaded: {len(df)} epochs")

    all_loss = [c for c in df.columns if "loss" in c.lower()]
    train_loss = [c for c in df.columns if "train" in c.lower() and "loss" in c.lower()]
    val_loss = [c for c in df.columns if "val" in c.lower() and "loss" in c.lower()]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Stage 1: Feature Extraction — Training Analysis", fontsize=14, fontweight="bold")

    # Plot 1: All losses
    ax = axes[0, 0]
    for col in all_loss:
        ax.plot(df[col], label=col, alpha=0.8)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("All Loss Curves"); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    # Plot 2: Train vs Val loss
    ax = axes[0, 1]
    if train_loss:
        ax.plot(df[train_loss[0]], label="Train Loss", color="blue", linewidth=2)
    if val_loss:
        ax.plot(df[val_loss[0]], label="Val Loss", color="red", linewidth=2)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("Train vs Val Loss (Overfitting Check)")
    ax.legend(); ax.grid(True, alpha=0.3)

    # Overfitting annotation
    if train_loss and val_loss:
        gap = df[val_loss[0]].values - df[train_loss[0]].values
        final_gap = gap[-1]
        color = "red" if final_gap > 0.5 else "green"
        label = f"OVERFITTING Gap={final_gap:.2f}" if final_gap > 0.5 else f"OK Gap={final_gap:.2f}"
        ax.annotate(label, xy=(len(gap)-1, df[val_loss[0]].values[-1]),
                   fontsize=10, color=color, fontweight="bold")

    # Plot 3: mAP
    ax = axes[1, 0]
    map50_col = [c for c in df.columns if "map50" in c.lower() and "map50-" not in c.lower()]
    map_col = [c for c in df.columns if "map50-95" in c.lower()]
    if map50_col:
        ax.plot(df[map50_col[0]], label="mAP@0.5", color="blue", linewidth=2)
    if map_col:
        ax.plot(df[map_col[0]], label="mAP@0.5:0.95", color="red", linewidth=2)
    ax.set_xlabel("Epoch"); ax.set_ylabel("mAP")
    ax.set_title("mAP Progress"); ax.legend(); ax.grid(True, alpha=0.3)

    # Plot 4: Overfitting gap
    ax = axes[1, 1]
    if train_loss and val_loss:
        gap = df[val_loss[0]].values - df[train_loss[0]].values
        ax.plot(gap, color="purple", linewidth=2)
        ax.axhline(y=0, color="green", linestyle="--", alpha=0.5, label="Perfect fit")
        ax.fill_between(range(len(gap)), 0, gap, alpha=0.3,
                       color="red" if gap[-1] > 0 else "green")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Val - Train Loss")
        ax.set_title("Overfitting Gap"); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(EVALS_DIR / "yolov26n_tl_stage1_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {EVALS_DIR / 'yolov26n_tl_stage1_curves.png'}")

print("\n" + "=" * 70)

  STAGE 1: TRAINING CURVES
  Loaded: 100 epochs


<Figure size 1400x1000 with 4 Axes>

  Saved: D:\DigiSteel-Yolo\DigiSteel-YOLO\evals\yolov26n_tl_stage1_curves.png



---
## Stage 2: Fine-Tuning

**Strategy:** Freeze early backbone (low-level features), unfreeze neck + head (high-level features)

**Why:** Early layers learn universal features (edges, textures). Neck layers need to adapt multi-scale fusion to steel defect sizes. Low LR preserves useful pretrained weights.

**LR:** 0.0001 (10x lower than Stage 1) | **Epochs:** 300

In [13]:
# ============================================================
# Cell 8: Stage 2 — Unfreeze Top Layers & Fine-Tune
# ============================================================
from ultralytics import YOLO
import time, json

print("=" * 70)
print("  STAGE 2: FINE-TUNING")
print("  Freeze: Early backbone (0-K) | Train: Neck + Head (K+1 to end)")
print("=" * 70)

# Load Stage 1 best weights (continue from Stage 1, NOT fresh pretrained)
if not S1_BEST_PT.exists():
    print(f"  ERROR: Stage 1 weights not found: {S1_BEST_PT}")
    print("  Run Cell 5 first.")
else:
    s2_model = YOLO(str(S1_BEST_PT))
    print(f"  Loaded Stage 1 weights: {S1_BEST_PT}")

    # Determine freeze boundary
    layers = list(s2_model.model.model.children())
    detect_idx = len(layers) - 1
    for i, layer in enumerate(layers):
        if type(layer).__name__ == "Detect":
            detect_idx = i
            break

    # Freeze early backbone only (roughly first 40% of non-head layers)
    # This preserves low-level feature detectors (edges, textures) while
    # allowing the neck (multi-scale fusion) to adapt to steel defect scales
    FREEZE_UP_TO = max(0, int(detect_idx * 0.4))
    print(f"  Freeze boundary: layers 0-{FREEZE_UP_TO} (early backbone — edges, textures)")
    print(f"  Trainable:       layers {FREEZE_UP_TO+1}-{detect_idx} (neck + head — fusion & detection)")

    frozen_params = 0
    trainable_params = 0

    for name, param in s2_model.model.named_parameters():
        parts = name.split(".")
        if parts[0] == "model" and len(parts) > 1:
            try:
                layer_idx = int(parts[1])
            except ValueError:
                layer_idx = -1
            if layer_idx <= FREEZE_UP_TO:
                param.requires_grad = False
                frozen_params += param.numel()
            else:
                param.requires_grad = True
                trainable_params += param.numel()
        else:
            param.requires_grad = True
            trainable_params += param.numel()

    total_p = frozen_params + trainable_params
    print(f"\n  Frozen:     {frozen_params:>12,} params ({frozen_params/total_p*100:.1f}%)")
    print(f"  Trainable:  {trainable_params:>12,} params ({trainable_params/total_p*100:.1f}%)")

    # Verify per-layer status
    print(f"\n  {'Idx':<5} {'Type':<20} {'Params':>12} {'Status':>12}")
    print(f"  {'-' * 55}")
    for i, layer in enumerate(layers):
        ltype = type(layer).__name__
        nparams = sum(p.numel() for p in layer.parameters())
        is_frozen = all(not p.requires_grad for p in layer.parameters()) if list(layer.parameters()) else True
        status = "FROZEN" if is_frozen else "TRAINABLE"
        print(f"  {i:<5} {ltype:<20} {nparams:>12,} {status:>12}")

    # ── Training (low LR, more epochs) ──
    s2_overrides = {
        "data": str(DATA_YAML),
        "task": "detect",
        "epochs": 300,
        "patience": 80,
        "batch": 16,
        "imgsz": 800,
        "device": 0,
        "optimizer": "AdamW",
        "lr0": 0.0001,          # 10x lower than Stage 1
        "lrf": 0.0001,          # Constant LR (no decay)
        "momentum": 0.937,
        "weight_decay": 0.0005,
        "warmup_epochs": 3,
        "warmup_momentum": 0.8,
        "warmup_bias_lr": 0.1,
        "mosaic": 0.0,
        "mixup": 0.15,
        "degrees": 10.0,
        "translate": 0.2,
        "scale": 0.6,
        "shear": 5.0,
        "perspective": 0.0,
        "flipud": 0.5,
        "fliplr": 0.5,
        "hsv_h": 0.015,
        "hsv_s": 0.7,
        "hsv_v": 0.4,
        "erasing": 0.4,
        "cos_lr": False,         # Constant LR for fine-tuning
        "deterministic": True,
        "close_mosaic": 10,
        "amp": True,
        "seed": 42,
        "workers": 8,
        "save": True,
        "save_period": 25,
        "plots": True,
        "project": str(RUNS_DIR),
        "name": S2_RUN_NAME,
        "exist_ok": True,
        "verbose": True,
    }

    print(f"\n  Starting fine-tuning...")
    print(f"  Epochs: {s2_overrides['epochs']} | LR: {s2_overrides['lr0']} (constant) | Patience: {s2_overrides['patience']}")

    s2_start = time.time()
    s2_results = s2_model.train(**s2_overrides)
    s2_time = time.time() - s2_start

    # Save status
    with open(EVALS_DIR / "yolov26n_tl_stage2_status.json", "w") as f:
        json.dump({"training_time_seconds": s2_time, "frozen_params": frozen_params,
                   "trainable_params": trainable_params, "freeze_up_to": FREEZE_UP_TO}, f, indent=2)

    print(f"\n{'=' * 60}")
    print(f"  STAGE 2 COMPLETE")
    print(f"  Duration: {s2_time/3600:.1f} hours")
    print(f"  Weights:  {S2_BEST_PT}")
    print(f"{'=' * 60}")

  STAGE 2: FINE-TUNING
  Freeze: Early backbone (0-K) | Train: Neck + Head (K+1 to end)
  Loaded Stage 1 weights: D:\DigiSteel-Yolo\DigiSteel-YOLO\runs\detect\yolov26n_tl_stage1\weights\best.pt
  Freeze boundary: layers 0-9 (early backbone — edges, textures)
  Trainable:       layers 10-23 (neck + head — fusion & detection)

  Frozen:        1,115,744 params (44.5%)
  Trainable:     1,390,396 params (55.5%)

  Idx   Type                       Params       Status
  -------------------------------------------------------
  0     Conv                          464       FROZEN
  1     Conv                        4,672       FROZEN
  2     C3k2                        6,640       FROZEN
  3     Conv                       36,992       FROZEN
  4     C3k2                       26,080       FROZEN
  5     Conv                      147,712       FROZEN
  6     C3k2                       87,040       FROZEN
  7     Conv                      295,424       FROZEN
  8     C3k2                      3

In [14]:
# ============================================================
# Cell 9: Stage 2 — Evaluate on Test Set
# ============================================================
from ultralytics import YOLO
import json

print("=" * 70)
print("  STAGE 2: TEST SET EVALUATION")
print("=" * 70)

if not S2_BEST_PT.exists():
    print(f"  ERROR: Weights not found: {S2_BEST_PT}")
    print("  Run Cell 8 first.")
else:
    s2_best = YOLO(str(S2_BEST_PT))

    print(f"\n  Evaluating on TEST set...")
    s2_val = s2_best.val(
        data=str(DATA_YAML), split="test", imgsz=800,
        batch=16, device=0, plots=True, verbose=True,
    )

    s2_map50 = float(s2_val.box.map50)
    s2_map5095 = float(s2_val.box.map)
    s2_prec = float(s2_val.box.mp)
    s2_rec = float(s2_val.box.mr)

    class_names = s2_best.names
    s2_per_class = {}
    for cid, cname in class_names.items():
        if cid < len(s2_val.box.ap50):
            s2_per_class[cname] = float(s2_val.box.ap50[cid])

    print(f"\n{'=' * 60}")
    print("  STAGE 2 TEST RESULTS")
    print(f"{'=' * 60}")
    print(f"  mAP@0.5:      {s2_map50:.4f} ({s2_map50*100:.1f}%)")
    print(f"  mAP@0.5:0.95: {s2_map5095:.4f} ({s2_map5095*100:.1f}%)")
    print(f"  Precision:    {s2_prec:.4f} ({s2_prec*100:.1f}%)")
    print(f"  Recall:       {s2_rec:.4f} ({s2_rec*100:.1f}%)")
    print(f"\n  Per-class AP@0.5:")
    for cname in sorted(s2_per_class):
        bar = "█" * int(s2_per_class[cname] * 40)
        print(f"    {cname:<18} {s2_per_class[cname]*100:.1f}% {bar}")

    # Compare with Stage 1
    s1_path = EVALS_DIR / "yolov26n_tl_stage1_results.json"
    if s1_path.exists():
        with open(s1_path) as f:
            s1 = json.load(f)
        print(f"\n  vs Stage 1 (Feature Extraction):")
        d50 = (s2_map50 - s1["mAP50"]) * 100
        d95 = (s2_map5095 - s1["mAP50_95"]) * 100
        print(f"    mAP@0.5:   {s1['mAP50']*100:.1f}% → {s2_map50*100:.1f}%  ({'+' if d50>0 else ''}{d50:.1f}%)")
        print(f"    mAP@50:95: {s1['mAP50_95']*100:.1f}% → {s2_map5095*100:.1f}%  ({'+' if d95>0 else ''}{d95:.1f}%)")

    # Compare with baselines
    baseline_path = EVALS_DIR / "fresh_baseline_results.json"
    yolov26_full_path = EVALS_DIR / "yolov26n_neudet_results.json"

    if baseline_path.exists():
        with open(baseline_path) as f:
            bl = json.load(f)
        d = (s2_map50 - bl["map50"]) * 100
        print(f"\n  vs YOLO11n Baseline: {'+' if d>0 else ''}{d:.1f}% mAP@0.5")

    if yolov26_full_path.exists():
        with open(yolov26_full_path) as f:
            y26 = json.load(f)
        d = (s2_map50 - y26["map50"]) * 100
        print(f"  vs YOLO26n Full:     {'+' if d>0 else ''}{d:.1f}% mAP@0.5")

    # Save
    s2_eval = {
        "stage": "fine_tuning",
        "model": "yolo26n.pt",
        "mAP50": s2_map50,
        "mAP50_95": s2_map5095,
        "precision": s2_prec,
        "recall": s2_rec,
        "per_class_ap50": s2_per_class,
    }
    with open(EVALS_DIR / "yolov26n_tl_stage2_results.json", "w") as f:
        json.dump(s2_eval, f, indent=2)
    print(f"\n  Saved: {EVALS_DIR / 'yolov26n_tl_stage2_results.json'}")

print("\n" + "=" * 70)

  STAGE 2: TEST SET EVALUATION

  Evaluating on TEST set...
Ultralytics 8.4.83  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
YOLO26n summary (fused): 122 layers, 2,376,006 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2.91.5 MB/s, size: 17.7 KB)
val: Scanning D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\labels\test.cache... 166 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 166/166  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 3.3it/s 3.3s0.1s
                   all        166        371      0.717      0.722      0.756      0.441
               crazing         24         54       0.64      0.362      0.459      0.181
             inclusion         33         77      0.748      0.844      0.832       0.45
               patches         31         80      0.822      0.806      0.848      0.534
        pitted_surface         

In [15]:
# ============================================================
# Cell 10: Stage 2 — Training Curves & Overfitting Analysis
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd
import json

print("=" * 70)
print("  STAGE 2: TRAINING CURVES")
print("=" * 70)

if not S2_RESULTS_CSV.exists():
    print(f"  ERROR: results.csv not found: {S2_RESULTS_CSV}")
else:
    df = pd.read_csv(S2_RESULTS_CSV)
    df.columns = df.columns.str.strip()
    print(f"  Loaded: {len(df)} epochs")

    all_loss = [c for c in df.columns if "loss" in c.lower()]
    train_loss = [c for c in df.columns if "train" in c.lower() and "loss" in c.lower()]
    val_loss = [c for c in df.columns if "val" in c.lower() and "loss" in c.lower()]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Stage 2: Fine-Tuning — Training Analysis", fontsize=14, fontweight="bold")

    # Plot 1: All losses
    ax = axes[0, 0]
    for col in all_loss:
        ax.plot(df[col], label=col, alpha=0.8)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("All Loss Curves"); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    # Plot 2: Train vs Val loss
    ax = axes[0, 1]
    if train_loss:
        ax.plot(df[train_loss[0]], label="Train Loss", color="blue", linewidth=2)
    if val_loss:
        ax.plot(df[val_loss[0]], label="Val Loss", color="red", linewidth=2)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("Train vs Val Loss (Overfitting Check)")
    ax.legend(); ax.grid(True, alpha=0.3)

    if train_loss and val_loss:
        gap = df[val_loss[0]].values - df[train_loss[0]].values
        final_gap = gap[-1]
        color = "red" if final_gap > 0.5 else "green"
        label = f"OVERFITTING Gap={final_gap:.2f}" if final_gap > 0.5 else f"OK Gap={final_gap:.2f}"
        ax.annotate(label, xy=(len(gap)-1, df[val_loss[0]].values[-1]),
                   fontsize=10, color=color, fontweight="bold")

    # Plot 3: mAP with Stage 1 reference line
    ax = axes[1, 0]
    map50_col = [c for c in df.columns if "map50" in c.lower() and "map50-" not in c.lower()]
    map_col = [c for c in df.columns if "map50-95" in c.lower()]
    if map50_col:
        ax.plot(df[map50_col[0]], label="mAP@0.5 (Stage 2)", color="blue", linewidth=2)
    if map_col:
        ax.plot(df[map_col[0]], label="mAP@0.5:0.95 (Stage 2)", color="red", linewidth=2)

    # Add Stage 1 final mAP as reference
    s1_path = EVALS_DIR / "yolov26n_tl_stage1_results.json"
    if s1_path.exists():
        with open(s1_path) as f:
            s1 = json.load(f)
        ax.axhline(y=s1["mAP50"], color="blue", linestyle="--", alpha=0.5, label=f"Stage 1 mAP@0.5 ({s1['mAP50']*100:.1f}%)")
        ax.axhline(y=s1["mAP50_95"], color="red", linestyle="--", alpha=0.5, label=f"Stage 1 mAP@50:95 ({s1['mAP50_95']*100:.1f}%)")

    ax.set_xlabel("Epoch"); ax.set_ylabel("mAP")
    ax.set_title("mAP Progress (with Stage 1 reference)")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # Plot 4: Overfitting gap
    ax = axes[1, 1]
    if train_loss and val_loss:
        gap = df[val_loss[0]].values - df[train_loss[0]].values
        ax.plot(gap, color="purple", linewidth=2)
        ax.axhline(y=0, color="green", linestyle="--", alpha=0.5, label="Perfect fit")
        ax.fill_between(range(len(gap)), 0, gap, alpha=0.3,
                       color="red" if gap[-1] > 0 else "green")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Val - Train Loss")
        ax.set_title("Overfitting Gap"); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(EVALS_DIR / "yolov26n_tl_stage2_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {EVALS_DIR / 'yolov26n_tl_stage2_curves.png'}")

print("\n" + "=" * 70)

  STAGE 2: TRAINING CURVES
  Loaded: 261 epochs


<Figure size 1400x1000 with 4 Axes>

  Saved: D:\DigiSteel-Yolo\DigiSteel-YOLO\evals\yolov26n_tl_stage2_curves.png



In [17]:
# ============================================================
# Cell 11: Final Comparison — Stage 1 vs Stage 2 vs Baselines
# ============================================================
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("=" * 70)
print("  FINAL COMPARISON")
print("=" * 70)

# Load all results
s1_path = EVALS_DIR / "yolov26n_tl_stage1_results.json"
s2_path = EVALS_DIR / "yolov26n_tl_stage2_results.json"
bl_path = EVALS_DIR / "fresh_baseline_results.json"
y26_path = EVALS_DIR / "yolov26n_neudet_results.json"

s1 = json.load(open(s1_path)) if s1_path.exists() else None
s2 = json.load(open(s2_path)) if s2_path.exists() else None
bl = json.load(open(bl_path)) if bl_path.exists() else None
y26 = json.load(open(y26_path)) if y26_path.exists() else None

if s1 and s2:
    # ── Summary Table ──
    models = {}
    if bl:
        models["YOLO11n Baseline"] = {"mAP50": bl["map50"], "mAP50_95": bl["map50_95"],
                                       "Precision": bl["precision"], "Recall": bl["recall"]}
    if y26:
        models["YOLO26n Full FT"] = {"mAP50": y26["map50"], "mAP50_95": y26["map50_95"],
                                      "Precision": y26["precision"], "Recall": y26["recall"]}
    models["Stage 1 (FE)"] = {"mAP50": s1["mAP50"], "mAP50_95": s1["mAP50_95"],
                               "Precision": s1["precision"], "Recall": s1["recall"]}
    models["Stage 2 (FT)"] = {"mAP50": s2["mAP50"], "mAP50_95": s2["mAP50_95"],
                               "Precision": s2["precision"], "Recall": s2["recall"]}

    df_summary = pd.DataFrame(models).T
    df_summary = df_summary * 100  # Convert to percentage
    print(f"\n  Overall Metrics (%):")
    print(df_summary.to_string(float_format="%.1f"))

    # ── Visual: Overall metrics bar chart ──
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    labels = list(models.keys())
    metrics = ["mAP50", "mAP50_95", "Precision", "Recall"]
    x = np.arange(len(metrics))
    w = 0.8 / len(labels)
    colors = ["#95a5a6", "#e67e22", "#3498db", "#e74c3c"]

    ax = axes[0]
    for i, (name, vals) in enumerate(models.items()):
        v = [vals[m] * 100 for m in metrics]
        ax.bar(x + i * w, v, w, label=name, color=colors[i % len(colors)], edgecolor="black", linewidth=0.5)
    ax.set_ylabel("Score (%)"); ax.set_title("Overall Metrics Comparison", fontweight="bold")
    ax.set_xticks(x + w * (len(labels) - 1) / 2)
    ax.set_xticklabels(metrics)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis="y"); ax.set_ylim(0, 105)

    # ── Visual: Per-class comparison ──
    ax = axes[1]
    classes = sorted(s1["per_class_ap50"].keys())
    x2 = np.arange(len(classes))

    per_class_data = {}
    if bl and "per_class_ap50" in bl:
        per_class_data["YOLO11n Baseline"] = [bl["per_class_ap50"].get(c, 0) * 100 for c in classes]
    if y26 and "per_class_ap50" in y26:
        per_class_data["YOLO26n Full FT"] = [y26["per_class_ap50"].get(c, 0) * 100 for c in classes]
    per_class_data["Stage 1 (FE)"] = [s1["per_class_ap50"].get(c, 0) * 100 for c in classes]
    per_class_data["Stage 2 (FT)"] = [s2["per_class_ap50"].get(c, 0) * 100 for c in classes]
    w2 = 0.8 / max(len(per_class_data), 1)

    for i, (name, vals) in enumerate(per_class_data.items()):
        ax.bar(x2 + i * w2, vals, w2, label=name, color=colors[i % len(colors)], edgecolor="black", linewidth=0.5)
    ax.set_ylabel("AP@0.5 (%)"); ax.set_title("Per-Class AP@0.5 Comparison", fontweight="bold")
    ax.set_xticks(x2 + w2 * (len(per_class_data) - 1) / 2)
    ax.set_xticklabels(classes, rotation=30, ha="right", fontsize=9)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis="y"); ax.set_ylim(0, 105)

    plt.tight_layout()
    plt.savefig(EVALS_DIR / "yolov26n_tl_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── Per-class delta table ──
    print(f"\n  Per-class AP@0.5 — Stage 2 vs Stage 1:")
    print(f"  {'Class':<20} {'Stage 1':>10} {'Stage 2':>10} {'Delta':>10}")
    print(f"  {'-' * 52}")
    for c in classes:
        v1 = s1["per_class_ap50"].get(c, 0) * 100
        v2 = s2["per_class_ap50"].get(c, 0) * 100
        d = v2 - v1
        print(f"  {c:<20} {v1:>9.1f}% {v2:>9.1f}% {'+' if d>0 else ''}{d:>8.1f}%")

    # ── Verdict ──
    print(f"\n{'=' * 60}")
    print("  VERDICT")
    print(f"{'=' * 60}")
    s2_vs_s1 = (s2["mAP50"] - s1["mAP50"]) * 100
    print(f"  Stage 2 vs Stage 1:  {'+' if s2_vs_s1>0 else ''}{s2_vs_s1:.1f}% mAP@0.5")
    if bl:
        s2_vs_bl = (s2["mAP50"] - bl["map50"]) * 100
        print(f"  Stage 2 vs YOLO11n:  {'+' if s2_vs_bl>0 else ''}{s2_vs_bl:.1f}% mAP@0.5")
    if y26:
        s2_vs_y26 = (s2["mAP50"] - y26["map50"]) * 100
        print(f"  Stage 2 vs YOLO26n Full: {'+' if s2_vs_y26>0 else ''}{s2_vs_y26:.1f}% mAP@0.5")

    # Save combined comparison
    comparison = {
        "stage1": s1,
        "stage2": s2,
        "stage2_vs_stage1": {"mAP50_delta": s2["mAP50"] - s1["mAP50"], "mAP50_95_delta": s2["mAP50_95"] - s1["mAP50_95"]},
    }
    if bl:
        comparison["baseline_yolo11n"] = bl
        comparison["stage2_vs_baseline"] = {"mAP50_delta": s2["mAP50"] - bl["map50"]}
    if y26:
        comparison["baseline_yolo26n_full"] = y26
        comparison["stage2_vs_yolo26n_full"] = {"mAP50_delta": s2["mAP50"] - y26["map50"]}

    with open(EVALS_DIR / "yolov26n_transfer_learning_comparison.json", "w") as f:
        json.dump(comparison, f, indent=2)
    print(f"\n  Saved: {EVALS_DIR / 'yolov26n_transfer_learning_comparison.json'}")

else:
    print("\n  Results not found. Run Cells 5, 6, 8, 9 first.")

print("\n" + "=" * 70)

  FINAL COMPARISON

  Overall Metrics (%):
                  mAP50  mAP50_95  Precision  Recall
YOLO11n Baseline   78.8      45.2       71.9    76.3
YOLO26n Full FT    81.0      43.1       77.3    76.2
Stage 1 (FE)       74.4      42.3       64.0    73.2
Stage 2 (FT)       75.6      44.1       71.7    72.2


<Figure size 1600x600 with 2 Axes>

<Figure size 1600x600 with 2 Axes>


  Per-class AP@0.5 — Stage 2 vs Stage 1:
  Class                   Stage 1    Stage 2      Delta
  ----------------------------------------------------
  crazing                   42.9%      45.9% +     3.0%
  inclusion                 81.6%      83.2% +     1.6%
  patches                   86.0%      84.8%     -1.2%
  pitted_surface            70.7%      70.1%     -0.6%
  rolled-in_scale           70.8%      73.0% +     2.2%
  scratches                 94.4%      96.7% +     2.4%

  VERDICT
  Stage 2 vs Stage 1:  +1.2% mAP@0.5
  Stage 2 vs YOLO11n:  -3.1% mAP@0.5
  Stage 2 vs YOLO26n Full: -5.4% mAP@0.5

  Saved: D:\DigiSteel-Yolo\DigiSteel-YOLO\evals\yolov26n_transfer_learning_comparison.json

